# Notebook 06 - RAG Pipeline

This notebook runs the RAG pipeline that the project uses today, end to end.
It builds the pipeline from `src/rag/pipeline.py`, checks what actually got
loaded, tries retrieval on five queries, answers one question in full, then runs
the 10-query logged test and saves the log to `reports/rag_pipeline_test_log.json`.

The pipeline does four things:

1. Embeds the query with `BAAI/bge-m3`.
2. Searches a FAISS `IndexFlatIP` index that holds 727,138 chunk vectors.
3. Reranks the candidates with `cross-encoder/ms-marco-MiniLM-L-12-v2` and keeps the best few.
4. Sends the evidence to `openai/gpt-oss-120b` on Groq and appends a medical disclaimer.

Building the pipeline takes a few minutes on the first run, because it loads the
embedding model, the 3 GB FAISS index, the 511 MB chunk mapping and the reranker.
The cell after the imports will look stuck for a while; that is normal.


## 1. Imports and path setup

The notebook sits in `notebooks/`, so the repo root is one level up. We add the
repo root to `sys.path` so `src.*` and `config.*` import, and load `.env` so the
Groq keys are available.


In [1]:
import json
import os
import sys
import time
from pathlib import Path

# notebook lives in notebooks/, so the repo root is one level up
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

print("Repo root:", ROOT)
print("Python:", sys.version.split()[0])


Repo root: D:\Projects\Healthcare-RAG-Assistant
Python: 3.12.10


## 2. Build the pipeline

`build_rag_pipeline()` returns a cached `RAGPipeline`. It loads the embedding
model, the FAISS index, the chunk mapping, the optional category classifier and
the cross-encoder reranker, so this cell needs a few minutes on a cold start.

Facts about the current knowledge base:

- Index file: `data/embeddings/faiss_index/kb_index_flatip.faiss` - 727,138 vectors, `IndexFlatIP`, 1024 dimensions.
- Mapping file: `data/embeddings/faiss_index/kb_chunk_mapping.pkl` with the columns `chunk_id`, `doc_id`, `source`, `source_id`, `category`, `text_chunk`. There are no `question`, `context` or `answer` columns any more; the pipeline derives those from `text_chunk`.
- Embedding model: `BAAI/bge-m3`. Reranker: `cross-encoder/ms-marco-MiniLM-L-12-v2`.
- Settings in `config/settings.py`: `TOP_K=30`, `INJECT_K=5`, `MAX_CONTEXT_WORDS=250`, `USE_RERANKER=True`.
- Generation runs on Groq with `openai/gpt-oss-120b` (the judge model used elsewhere is `qwen/qwen3.8-27b`). `.env` holds 31 comma-separated `GROQ_API_KEY` values, and `src/rag/pipeline.py` loads `.env` at import time, so no extra setup is needed before `build_rag_pipeline()`.
- Retrieval is dense embedding search plus reranking only. No keyword index is built.


In [2]:
from src.rag.pipeline import build_rag_pipeline

start = time.time()
pipeline = build_rag_pipeline()
print(f"Pipeline built in {time.time() - start:.1f}s")


📥 Local model not found. Downloading from HuggingFace: AbdoMatrix/biobert-medical-classifier
✅ Classifier loaded | Device: cuda | Classes: ['Diagnosis', 'General', 'Medication', 'Prevention', 'Symptoms', 'Treatment']
Pipeline built in 37.1s


## 3. What got loaded

Read the numbers straight off the pipeline object instead of trusting the docs.


In [3]:
mapping = pipeline.mapping_df

print("Index type:", type(pipeline.index).__name__)
print("Index vectors:", f"{pipeline.index.ntotal:,}")
print("Index dimensions:", pipeline.index.d)
print("Mapping rows:", f"{len(mapping):,}")
print("Mapping columns:", mapping.columns.tolist())
print("Embedding model:", pipeline.encoder[0].auto_model.config.name_or_path)
print("top_k (candidates fetched):", pipeline.top_k)
print("inject_k (chunks sent to the LLM):", pipeline.inject_k)
print("max_context_words:", pipeline.max_context_words)
print("max_new_tokens:", pipeline.max_new_tokens)
print("Reranker enabled:", pipeline._use_reranker)
print("Reranker model:", pipeline.reranker.config._name_or_path if pipeline._use_reranker else "not loaded")
print("Groq client active:", pipeline._use_groq)
print("Groq model:", getattr(pipeline, "_groq_model", "n/a"))
print("Groq keys loaded:", len(getattr(pipeline, "_groq_clients", [])))
print("Classifier active:", pipeline._use_classifier)

print()
first = mapping.iloc[0]
print("First mapping row:")
print("  chunk_id:", first["chunk_id"], "| doc_id:", first["doc_id"], "| source:", first["source"],
      "| source_id:", first["source_id"], "| category:", first["category"])
print("  text_chunk (first 200 chars):", " ".join(str(first["text_chunk"]).split())[:200])


Index type: IndexFlatIP
Index vectors: 727,138
Index dimensions: 1024
Mapping rows: 727,138
Mapping columns: ['chunk_id', 'doc_id', 'source', 'source_id', 'category', 'text_chunk']
Embedding model: BAAI/bge-m3
top_k (candidates fetched): 30
inject_k (chunks sent to the LLM): 5
max_context_words: 250
max_new_tokens: 2048
Reranker enabled: True
Reranker model: cross-encoder/ms-marco-MiniLM-L-12-v2
Groq client active: True
Groq model: openai/gpt-oss-120b
Groq keys loaded: 31
Classifier active: True

First mapping row:
  chunk_id: 0 | doc_id: 0 | source: medquad | source_id: 3005 | category: treatment
  text_chunk (first 200 chars): Question: What are the treatments for Breast Cancer? Answer: Key Points - There are different types of treatment for patients with breast cancer. - Five types of standard treatment are used: - Surgery


## 4. Retrieval test

Five queries, one per category. Each `retrieve` call fetches `top_k` candidates,
reranks them and returns the best five, so we print the chunk id, the source, the
reranker score and a short excerpt of the chunk text.


In [4]:
category_queries = {
    "Symptoms": "What are the early symptoms of type 2 diabetes?",
    "Diagnosis": "How is pneumonia diagnosed in elderly patients?",
    "Treatment": "What are the current treatment options for hypertension?",
    "Medication": "What are the side effects of metformin?",
    "Prevention": "How can cardiovascular disease be prevented through lifestyle changes?",
}

for category, query in category_queries.items():
    chunks = pipeline.retrieve(query, top_k=5)
    print("=" * 95)
    print(f"{category}: {query}")
    for rank, chunk in enumerate(chunks, 1):
        excerpt = " ".join(chunk["text_chunk"].split())[:130]
        print(f"  {rank}. chunk_id={chunk['chunk_id']} source={chunk['source']} "
              f"reranker_score={chunk['reranker_score']:.3f} distance={chunk['distance']:.3f}")
        print(f"     {excerpt}")
    print()


Symptoms: What are the early symptoms of type 2 diabetes?
  1. chunk_id=23076 source=medquad reranker_score=8.540 distance=0.751
     Question: What are the symptoms of Am I at Risk for Type 2 Diabetes? Taking Steps to Lower Your Risk of Getting Diabetes? Answer: 
  2. chunk_id=17938 source=medquad reranker_score=7.805 distance=0.742
     Question: What is (are) Diabetes Type 2? . The symptoms of type 2 diabetes appear slowly. Some people do not notice symptoms at al
  3. chunk_id=4782 source=medquad reranker_score=6.713 distance=0.751
     Question: What are the symptoms of Diabetes? Answer: Diabetes is often called a "silent" disease because it can cause serious comp
  4. chunk_id=25940 source=medquad reranker_score=6.437 distance=0.732
     Question: What are the symptoms of Your Guide to Diabetes: Type 1 and Type 2? Answer: The signs and symptoms of diabetes are - bei
  5. chunk_id=29975 source=medlineplus reranker_score=4.822 distance=0.739
     Question: What is Diabetes Type 2? 

## 5. Full pipeline on one query

`pipeline.answer(query)` runs the whole chain: routing guard, retrieval,
reranking, generation and the disclaimer. It returns a dict, and we print the
fields we care about.


In [5]:
result = pipeline.answer("What are the symptoms of diabetes?")

print("Question:", result["question"])
print()
print("Answer:")
print(result["answer"])
print()
print("answer_source:", result["answer_source"])
print("used_rag:", result["used_rag"])
print("disclaimer_present:", result["disclaimer_present"])
print("sources returned:", len(result["retrieved_sources"]))
print("top_k:", result["top_k"])
print("retrieval_quality (mean reranker score):", result["retrieval_quality"])
print("mean_cosine_similarity:", result["mean_cosine_similarity"])
print()
print("First three sources:")
for source in result["retrieved_sources"][:3]:
    print(f"  chunk_id={source['chunk_id']} category={source['category']} "
          f"reranker_score={source['reranker_score']} cosine={source['distance']}")


Question: What are the symptoms of diabetes?

Answer:
Diabetes often shows up with several noticeable signs that affect how you feel and how your body works. • feeling very thirsty – you may want to drink water all the time • needing to urinate often, especially at night – you go to the bathroom more frequently • feeling very hungry even after eating – a constant urge to eat • feeling very tired or weak – low energy throughout the day • losing weight without trying – you drop pounds even though you eat normally • blurry or fuzzy vision – things don’t look as clear as they used to • dry, itchy skin – skin feels rough and may itch a lot • sores or cuts that heal slowly – wounds take longer to close • tingling, numbness, or a “pins‑and‑needles” feeling in the feet or hands – unusual sensations in the extremities Some people with diabetes have only a few of these signs or none at all, so the only reliable way to know if you have diabetes is to have a blood test done by a health professiona

## 6. Ten-query logged run

The same 10 test queries the notebook used before, now recorded with the fields
the current pipeline returns. For each query we print the latency, where the
answer came from and the first 200 characters of the answer, and we keep the full
dicts in `results` for the summary and the log file.


In [6]:
test_queries = [
    # Symptoms
    "What are the early symptoms of type 2 diabetes?",
    # Diagnosis
    "How is pneumonia diagnosed in elderly patients?",
    # Treatment
    "What are the current treatment options for hypertension?",
    # Medication
    "What are the side effects of metformin?",
    # Prevention
    "How can cardiovascular disease be prevented through lifestyle changes?",
    # General
    "What is the role of the immune system in fighting infections?",
    # Mixed / clinical
    "Is laparoscopic surgery better than open surgery for prostatectomy?",
    "Can bacterial gastroenteritis lead to irritable bowel syndrome?",
    "Does naturopathy effectively treat menopausal symptoms?",
    "Is urgent colonoscopy needed for acute diverticular bleeding?",
]

print(f"{len(test_queries)} test queries ready")
for number, query in enumerate(test_queries, 1):
    print(f"  {number}. {query}")


10 test queries ready
  1. What are the early symptoms of type 2 diabetes?
  2. How is pneumonia diagnosed in elderly patients?
  3. What are the current treatment options for hypertension?
  4. What are the side effects of metformin?
  5. How can cardiovascular disease be prevented through lifestyle changes?
  6. What is the role of the immune system in fighting infections?
  7. Is laparoscopic surgery better than open surgery for prostatectomy?
  8. Can bacterial gastroenteritis lead to irritable bowel syndrome?
  9. Does naturopathy effectively treat menopausal symptoms?
  10. Is urgent colonoscopy needed for acute diverticular bleeding?


In [7]:
results = []
latencies = []


def json_safe_sources(sources):
    """Some rows in the mapping have no category (pandas NA), which json cannot write,
    so each source is rebuilt with plain Python types before it goes into the log."""
    records = []
    for source in sources:
        category = source["category"] if isinstance(source["category"], str) else ""
        records.append({
            "chunk_id": int(source["chunk_id"]),
            "question": str(source["question"]),
            "category": category,
            "category_score": float(source["category_score"]),
            "distance": float(source["distance"]),
            "relevance_score": float(source["relevance_score"]),
            "reranker_score": float(source["reranker_score"]),
            "context": str(source["context"]),
            "answer": str(source["answer"]),
            "excerpt": str(source["excerpt"]),
        })
    return records


print("=" * 95)
print("10-QUERY LOGGED RUN")
print("=" * 95)

for number, query in enumerate(test_queries, 1):
    start = time.time()
    result = pipeline.answer(query)
    elapsed_ms = (time.time() - start) * 1000
    latencies.append(elapsed_ms)

    results.append({
        "query_num": number,
        "question": result["question"],
        "answer_raw": result["answer_raw"],
        "disclaimer_present": result["disclaimer_present"],
        "top_k": result["top_k"],
        "used_rag": result["used_rag"],
        "answer_source": result["answer_source"],
        "retrieval_quality": result["retrieval_quality"],
        "mean_cosine_similarity": result["mean_cosine_similarity"],
        "sources": json_safe_sources(result["retrieved_sources"]),
        "latency_ms": round(elapsed_ms, 2),
    })

    print()
    print("-" * 95)
    print(f"Query {number}: {query}")
    print(f"Latency: {elapsed_ms:.0f} ms")
    print(f"Sources: {result['top_k']} | answer_source: {result['answer_source']} | used_rag: {result['used_rag']}")
    print(f"Source chunk ids: {[s['chunk_id'] for s in result['retrieved_sources']]}")
    print(f"Answer (first 200 chars): {result['answer_raw'][:200]}")


10-QUERY LOGGED RUN

-----------------------------------------------------------------------------------------------
Query 1: What are the early symptoms of type 2 diabetes?
Latency: 25678 ms
Sources: 30 | answer_source: rag | used_rag: True
Source chunk ids: [23076, 17938, 4786, 4782, 3483, 25940, 26667, 1529, 10134, 29948, 29975, 17937, 29971, 29973, 5886, 4321, 26763, 7022, 685054, 10008, 4783, 7982, 6874, 13468, 21572, 16352, 670885, 4785, 10969, 3082]
Answer (first 200 chars): Early signs of type 2 diabetes often appear slowly and can be easy to miss. Common early symptoms include: • Feeling very thirsty even when you drink a lot • Going to the bathroom to urinate more ofte

-----------------------------------------------------------------------------------------------
Query 2: How is pneumonia diagnosed in elderly patients?
Latency: 19986 ms
Sources: 30 | answer_source: rag | used_rag: True
Source chunk ids: [278458, 608848, 31264, 608850, 83784, 83783, 4809, 449140, 83785, 27846

## 7. Summary and log

Counts, latency stats and how many answers came from the retrieved evidence
versus the model's general knowledge. Then we save the run to
`reports/rag_pipeline_test_log.json`.


In [8]:
mean_latency = sum(latencies) / len(latencies)
rag_answers = sum(1 for r in results if r["answer_source"] == "rag")
general_answers = sum(1 for r in results if r["answer_source"] == "general_knowledge")

print("=" * 60)
print("10-QUERY TEST SUMMARY")
print("=" * 60)
print(f"Queries tested:                 {len(results)}")
print(f"Disclaimer in every answer:     {all(r['disclaimer_present'] for r in results)}")
print(f"Sources retrieved per query:    {[r['top_k'] for r in results]}")
print(f"Answers from retrieved evidence: {rag_answers}")
print(f"Answers from general knowledge:  {general_answers}")
print()
print("Latency (ms):")
print(f"  min:   {min(latencies):.0f}")
print(f"  max:   {max(latencies):.0f}")
print(f"  mean:  {mean_latency:.0f}")
print(f"  total: {sum(latencies) / 1000:.1f} s")
print()
print("Mean retrieval quality (reranker score): "
      f"{sum(r['retrieval_quality'] for r in results) / len(results):.3f}")
print("Mean cosine similarity of retrieved chunks: "
      f"{sum(r['mean_cosine_similarity'] for r in results) / len(results):.3f}")

for r in results:
    assert r["disclaimer_present"], f"missing disclaimer for: {r['question']}"

print()
print("Every query returned a disclaimer.")


10-QUERY TEST SUMMARY
Queries tested:                 10
Disclaimer in every answer:     True
Sources retrieved per query:    [30, 30, 30, 30, 30, 30, 30, 30, 30, 30]
Answers from retrieved evidence: 7
Answers from general knowledge:  3

Latency (ms):
  min:   15741
  max:   25678
  mean:  20946
  total: 209.5 s

Mean retrieval quality (reranker score): 0.035
Mean cosine similarity of retrieved chunks: 0.639

Every query returned a disclaimer.


In [9]:
log_path = ROOT / "reports" / "rag_pipeline_test_log.json"
tmp_path = log_path.with_suffix(".json.tmp")

# write next to the target and rename, so an interrupted run cannot leave a half-written log
with open(tmp_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
os.replace(tmp_path, log_path)

print("Test log saved to:", log_path)
print("Queries in log:", len(results))
print("Sources per query:", [len(r["sources"]) for r in results])
print("Answer sources:", [r["answer_source"] for r in results])


Test log saved to: D:\Projects\Healthcare-RAG-Assistant\reports\rag_pipeline_test_log.json
Queries in log: 10
Sources per query: [30, 30, 30, 30, 30, 30, 30, 30, 30, 30]
Answer sources: ['rag', 'rag', 'general_knowledge', 'rag', 'rag', 'rag', 'general_knowledge', 'rag', 'general_knowledge', 'rag']


## Summary

| Item | Value |
|---|---|
| Embedding model | `BAAI/bge-m3` (1024-dim) |
| Vector store | FAISS `IndexFlatIP`, 727,138 vectors |
| Reranker | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| Retrieval depth | `TOP_K=30` candidates, `INJECT_K=5` chunks to the LLM |
| Generator | `openai/gpt-oss-120b` on Groq |
| Disclaimer | Appended to every answer |
| Test run | 10 queries with latency and sources, saved to `reports/rag_pipeline_test_log.json` |

Next notebook: `07_classification_model.ipynb` for the category classifier.
